<a href="https://colab.research.google.com/github/KrishDataLab/ML_Narchukoora_babu/blob/main/Classification_task_using_RF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hotel booking demand
From the paper: hotel booking demand datasets

In [3]:
import kagglehub
path = kagglehub.dataset_download("jessemostipak/hotel-booking-demand")

Using Colab cache for faster access to the 'hotel-booking-demand' dataset.


In [23]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score


**Load** **Data**

# 1. Classification Tasks (Most Popular)
A. Booking Cancellation Prediction (Primary Task)
Target Variable: is_canceled (0 = Not canceled, 1 = Canceled)

Goal: Predict whether a customer will cancel their hotel reservation before arrival.

Why it matters: Helps hotel management optimize overbooking strategies, minimize revenue loss, and apply dynamic deposit policies for high-risk bookings.

Recommended Models: Logistic Regression, Random Forest, XGBoost, LightGBM.

Key Features: lead_time, deposit_type, market_segment, previous_cancellations, total_of_special_requests, booking_changes.

In [12]:
import os

csv_file_name = 'hotel_bookings.csv'
full_csv_path = os.path.join(path, csv_file_name)
df = pd.read_csv(full_csv_path)

In [13]:
df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [16]:
df.shape

(119390, 32)

In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  object 
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  object 
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal            

In [19]:
df.columns

Index(['hotel', 'is_canceled', 'lead_time', 'arrival_date_year',
       'arrival_date_month', 'arrival_date_week_number',
       'arrival_date_day_of_month', 'stays_in_weekend_nights',
       'stays_in_week_nights', 'adults', 'children', 'babies', 'meal',
       'country', 'market_segment', 'distribution_channel',
       'is_repeated_guest', 'previous_cancellations',
       'previous_bookings_not_canceled', 'reserved_room_type',
       'assigned_room_type', 'booking_changes', 'deposit_type', 'agent',
       'company', 'days_in_waiting_list', 'customer_type', 'adr',
       'required_car_parking_spaces', 'total_of_special_requests',
       'reservation_status', 'reservation_status_date'],
      dtype='object')

In [20]:
df_cleaned = df.copy()

In [28]:
# Identify numerical and categorical columns
X = df_cleaned.drop(columns=['is_canceled', 'reservation_status', 'reservation_status_date'])
y = df_cleaned['is_canceled']

In [27]:
# Identify numerical and categorical columns
num_cols = X.select_dtypes(include=['int64', 'float64', 'int32']).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

In [29]:
# 2. Train / Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
    )

In [30]:
# 3. Preprocessing Pipeline
preprocessor = ColumnTransformer(
    transformers=[
            ('num', StandardScaler(), num_cols),
                    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
                        ]
                        )

In [31]:
# 4. Model Pipeline (Random Forest Classifier)
model_rf = Pipeline(steps=[
    ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
        ])

In [32]:
# 5. Train Model
model_rf.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['lead_time',
                                                   'arrival_date_year',
                                                   'arrival_date_week_number',
                                                   'arrival_date_day_of_month',
                                                   'stays_in_weekend_nights',
                                                   'stays_in_week_nights',
                                                   'adults', 'children',
                                                   'babies',
                                                   'is_repeated_guest',
                                                   'previous_cancellations',
                                                   'previous_bookings_not_canceled',
                                                   'booking_changes',...
                                                   'total_of_special_requests',
                                                   'total_stay_nights',
                                                   'total_guests']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['hotel',
                                                   'arrival_date_month', 'meal',
                                                   'country', 'market_segment',
                                                   'distribution_channel',
                                                   'reserved_room_type',
                                                   'assigned_room_type',
                                                   'deposit_type',
                                                   'customer_type'])])),
                ('classifier',
                 RandomForestClassifier(n_jobs=-1, random_state=42))])

In [33]:
# 6. Evaluation
y_pred = model_rf.predict(X_test)
y_proba = model_rf.predict_proba(X_test)[:, 1]

print("=== Classification Report ===")
print(classification_report(y_test, y_pred))

print("=== ROC-AUC Score ===")
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}")

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.87      0.93      0.90     12644
           1       0.78      0.64      0.70      4802

    accuracy                           0.85     17446
   macro avg       0.82      0.78      0.80     17446
weighted avg       0.85      0.85      0.85     17446

=== ROC-AUC Score ===
ROC-AUC: 0.9098
